In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

df = pd.read_csv('placement_predict_50k_adjusted.csv')
df = df.drop(columns=['IsAnomaly'])

for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].fillna(df[col].median())
    else:
        df[col] = df[col].fillna(df[col].mode()[0])

df_encoded = pd.get_dummies(
    df,
    columns=[
        'Gender', 'City', 'CollegeTier', 'Stream',
        'Specialisation', 'Hostel', 'HistoryOfBacklogs',
        'ExtraCurricular'
    ],
    drop_first=True
)

X = df_encoded.drop(columns=['PlacementStatus'])
y = df_encoded['PlacementStatus']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

plt.figure(figsize=(20, 10))
plot_tree(
    model,
    feature_names=X.columns,
    class_names=['Not Placed', 'Placed'],
    filled=True,
    fontsize=7
)
plt.tight_layout()
plt.savefig('decision_tree.png', dpi=150)
plt.show()

from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42

df = pd.read_csv("simple_placement_dataset.csv")

print(df.head())
print(f"\nDataset shape: {df.shape}")
print(f"Class balance:\n{df['Placed'].value_counts()}")

TARGET = "Placed"
DROP_COLS = ["StudentID"]

feature_names = [
    c for c in df.columns
    if c not in [TARGET] + DROP_COLS
]

X = df[feature_names].values
y = df[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

rf = RandomForestClassifier(
    n_estimators=100,
    max_features="sqrt",
    oob_score=True,
    random_state=RANDOM_STATE
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)

print(f"\nOOB score: {rf.oob_score_:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred))

print("Feature importances:")
for name, importance in sorted(
    zip(feature_names, rf.feature_importances_),
    key=lambda x: -x[1]
):
    print(f" {name}: {importance:.4f}")
